# 03 · Inserción de datos
### Proyecto Olist · Python → MySQL → Tableau

---

## ¿Por qué Python para la inserción?

Cada herramienta tiene su propósito. SQL sirve para crear tablas y consultar datos. Python sirve para trabajar con archivos y automatizar tareas repetitivas — por eso es la opción natural para leer los CSV e insertarlos en la base de datos.

Workbench tiene una opción para importar archivos (Table Data Import Wizard), pero hay que usarlo cada vez por cada archivo, el proceso es lento con volúmenes grandes y cualquier error obliga a empezar de cero. Con Python, la inserción de los datos se ejecuta en segundos con un solo click.

---

## Objetivo de este notebook

Insertar los 5 CSV limpios en MySQL respetando el orden de las claves foráneas.  
Requiere haber ejecutado `02_creacion_tablas.sql` en Workbench.

---

## Orden de inserción

Las claves foráneas obligan a un orden — una tabla no puede referenciar otra que aún no tiene datos:

`customers · products → orders → order_items · reviews`

---
# Librerías

In [36]:
import os                       # os → leer las variables del archivo .env
from dotenv import load_dotenv  # load_dotenv → leer el archivo .env con las credenciales
import pandas as pd             # pandas → cargar los CSV antes de insertarlos
import mysql.connector          # mysql.connector → conectar Python con MySQL

---
# 1️⃣ Conexión a MySQL

In [ ]:
# load_dotenv() lo lee y lo deja disponible con os.getenv()
load_dotenv()

# try/except → si algo falla (contraseña incorrecta, MySQL apagado...)
# imprime el error en lugar de romper el notebook sin explicación
try:
    cnx = mysql.connector.connect(
    host     = os.getenv('MYSQL_HOST'),
    user     = os.getenv('MYSQL_USER'),
    password = os.getenv('MYSQL_PASSWORD'),
    database = "olist",                      
    auth_plugin='mysql_native_password',
)
    print('✅ Conexión exitosa')

except mysql.connector.Error as e:
    print('❌ Error al conectar:', e)

✅ Conexión exitosa


In [42]:
# Para enviar consultas SQL desde Python necesitamos dos cosas:
# cnx → la conexión con MySQL 
# cursor → el objeto que envía y ejecuta las consultas dentro de esa conexión 
# Sin cursor no podemos ejecutar nada, aunque la conexión esté abierta
cursor = cnx.cursor()

print('✅ Cursor creado correctamente')

✅ Cursor creado correctamente


---
# 2️⃣ Inserción de datos

Flujo por tabla: **leer CSV → convertir NaN a None → insertar → confirmar**

In [43]:
output_path = "output"  # carpeta con los CSV limpios generados en 01_limpieza_olist.ipynb

In [44]:
## 1) CUSTOMERS
customers = pd.read_csv(os.path.join(output_path, "customers_clean.csv"))  # abre el CSV como tabla
customers = customers.replace({pd.NA: None, float('nan'): None})  # pandas usa NaN, MySQL solo acepta NULL

cursor.executemany(
    "INSERT IGNORE INTO customers (customer_id, customer_state) VALUES (%s, %s)",
    list(customers.itertuples(index=False, name=None))
)
cnx.commit()
print(f"✅ customers: {len(customers):,} registros insertados")

✅ customers: 99,441 registros insertados


In [45]:
## 2) PRODUCTS
products = pd.read_csv(os.path.join(output_path, "products_clean.csv"))
products = products.replace({pd.NA: None, float('nan'): None})

cursor.executemany(
    "INSERT IGNORE INTO products (product_id, product_category_name) VALUES (%s, %s)",
    list(products.itertuples(index=False, name=None))
)
cnx.commit()
print(f"✅ products: {len(products):,} registros insertados")

✅ products: 32,951 registros insertados


In [46]:
## 3) ORDERS
# la fecha necesita formato MySQL antes de insertar
orders = pd.read_csv(os.path.join(output_path, "orders_clean.csv"))
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp']).dt.strftime('%Y-%m-%d %H:%M:%S')
orders = orders.replace({pd.NA: None, float('nan'): None})

cursor.executemany(
    "INSERT IGNORE INTO orders (order_id, customer_id, order_purchase_timestamp) VALUES (%s, %s, %s)",
    list(orders.itertuples(index=False, name=None))
)
cnx.commit()
print(f"✅ orders: {len(orders):,} registros insertados")

✅ orders: 96,478 registros insertados


In [47]:
## 4) ORDER_ITEMS
order_items = pd.read_csv(os.path.join(output_path, "order_items_clean.csv"))
order_items = order_items.replace({pd.NA: None, float('nan'): None})

cursor.executemany(
    "INSERT IGNORE INTO order_items (order_id, product_id, price) VALUES (%s, %s, %s)",
    list(order_items.itertuples(index=False, name=None))
)
cnx.commit()
print(f"✅ order_items: {len(order_items):,} registros insertados")

✅ order_items: 112,650 registros insertados


In [48]:
## 5) REVIEWS
reviews = pd.read_csv(os.path.join(output_path, "reviews_clean.csv"))
reviews = reviews.replace({pd.NA: None, float('nan'): None})

cursor.executemany(
    "INSERT IGNORE INTO reviews (order_id, review_score) VALUES (%s, %s)",
    list(reviews.itertuples(index=False, name=None))
)
cnx.commit()
print(f"✅ reviews: {len(reviews):,} registros insertados")

✅ reviews: 98,673 registros insertados


---
# 3️⃣ Verificación

`diff = 0` → todo correcto · `diff > 0` → datos descartados por SQL

In [49]:
tablas = [
    ("customers",   customers),
    ("products",    products),
    ("orders",      orders),
    ("order_items", order_items),
    ("reviews",     reviews),
]

print(f"{'Tabla':<15} {'CSV':>8} {'MySQL':>8} {'Diff':>6}")
print("-" * 42)

for nombre_tabla, dataframe in tablas:
    cursor.execute(f"SELECT COUNT(*) FROM {nombre_tabla}")
    registros_mysql = cursor.fetchone()[0]
    registros_csv   = len(dataframe)
    diferencia      = registros_csv - registros_mysql
    estado          = "✅" if diferencia == 0 else f"⚠️  -{diferencia:,}"
    print(f"{nombre_tabla:<15} {registros_csv:>8,} {registros_mysql:>8,} {estado:>6}")

Tabla                CSV    MySQL   Diff
------------------------------------------
customers         99,441   99,441      ✅
products          32,951   32,951      ✅
orders            96,478   96,478      ✅
order_items      112,650  100,196 ⚠️  -12,454
reviews           98,673   95,832 ⚠️  -2,841


---
# 4️⃣ Cerrar conexión

In [ ]:
cursor.close()
cnx.close()
print("🔒 Conexión cerrada")
print("✅ Siguiente: 04_consultas.sql en Workbench.")